# Prompt Engineering Portfolio — Week 1
### GenAI Roadmap — Week 1 Mini-Project

Demonstrates **12 prompt engineering techniques** applied across four task types — text classification, summarization, code generation, and data extraction — with outputs compared across **OpenAI GPT**, **Anthropic Claude**, and a **local Ollama model**.

| # | Technique | Task type |
|---|-----------|-----------|
| 1 | Zero-Shot Prompting | Text classification |
| 2 | Few-Shot Prompting | Text classification |
| 3 | Chain-of-Thought (CoT) | Data extraction / reasoning |
| 4 | System Message & Role Assignment | Summarization |
| 5 | Structured Output (JSON) | Data extraction |
| 6 | ReAct Pattern (reason + act with a tool) | Reasoning / tool use |
| 7 | Prompt Chaining | Summarization |
| 8 | Code Generation Prompting | Code generation |
| 9 | Reflection / Iterative Refinement | Code generation |
| 10 | Role-Playing / Persona Prompting | Text generation |
| 11 | Self-Consistency (sample & vote) | Reasoning |
| 12 | Cross-Model Comparison | All tasks |

## Setup

1. Copy `.env.example` to `.env` in this folder and fill in your keys:
   - `OPENAI_API_KEY`
   - `ANTHROPIC_API_KEY`
2. Install dependencies: `pip install -r requirements.txt`
3. For the local model, install [Ollama](https://ollama.ai), run `ollama serve`, then pull a model:
   `ollama pull llama3.1`

In [ ]:
import os
import re
import ast
import operator
import json
from collections import Counter

from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1")

In [ ]:
from openai import OpenAI
import anthropic
import ollama

openai_client = OpenAI(api_key=OPENAI_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def ask_gpt(prompt, system=None, model="gpt-4o-mini", temperature=0.7):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = openai_client.chat.completions.create(
        model=model, messages=messages, temperature=temperature,
    )
    return response.choices[0].message.content


def ask_claude(prompt, system=None, model="claude-sonnet-5", temperature=0.7, max_tokens=1024):
    response = anthropic_client.messages.create(
        model=model,
        system=system or "",
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


def ask_ollama(prompt, system=None, model=None, temperature=0.7):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = ollama.chat(
        model=model or OLLAMA_MODEL, messages=messages, options={"temperature": temperature},
    )
    return response["message"]["content"]


def compare_models(prompt, system=None):
    return {
        "GPT": ask_gpt(prompt, system=system),
        "Claude": ask_claude(prompt, system=system),
        "Ollama": ask_ollama(prompt, system=system),
    }


def show(results):
    for name, text in results.items():
        print(f"--- {name} ---\n{text}\n")

## 1. Zero-Shot Prompting
*Task: Text classification.* Direct instructions, no examples — the baseline for every task.

In [ ]:
review = (
    "The new update completely broke my workflow, but customer support fixed it "
    "within an hour and even followed up the next day."
)

zero_shot_prompt = f"""Classify the sentiment of the following review as Positive, Negative, or Mixed. Respond with a single word.

Review: {review}"""

show(compare_models(zero_shot_prompt))

## 2. Few-Shot Prompting
*Task: Text classification.* A handful of labeled examples steers the format and category boundaries.

In [ ]:
few_shot_prompt = """Classify each support ticket into one category: Billing, Technical, Account, or General.

Ticket: \"I was charged twice for my subscription this month.\"
Category: Billing

Ticket: \"The app crashes every time I try to upload a photo.\"
Category: Technical

Ticket: \"How do I change the email linked to my account?\"
Category: Account

Ticket: \"Do you have a mobile app for iOS?\"
Category: General

Ticket: \"My invoice shows a $15 charge I don't recognize.\"
Category:"""

show(compare_models(few_shot_prompt))

## 3. Chain-of-Thought (CoT) Prompting
*Task: Data extraction / arithmetic reasoning.* Explicit "think step by step" reasoning before the final answer.

In [ ]:
cot_prompt = """A customer order says: \"2 large pizzas at $14.50 each, 3 sodas at $2.25 each, and a $5 delivery fee. A 10% discount code was applied to the food subtotal only.\"

Let's think step by step to calculate the final total, then give the final answer on the last line as: Final Total: $X.XX"""

show(compare_models(cot_prompt))

## 4. System Message & Role Assignment
*Task: Summarization.* A system prompt sets persona and constraints that shape every response.

In [ ]:
system_msg = (
    "You are a senior technical editor at a software company. You write concise, "
    "jargon-free summaries for non-technical executives."
)

article = (
    "Retrieval-Augmented Generation (RAG) is a technique that lets a language model answer questions "
    "using information it was never trained on. Instead of relying only on what the model memorized "
    "during training, the system first searches an external knowledge base — usually a vector database "
    "of document embeddings — for the passages most relevant to the user's query. Those passages are "
    "then inserted into the prompt as context before the model generates its answer. This approach "
    "reduces hallucination, keeps answers grounded in a specific, updatable source of truth, and avoids "
    "the cost of retraining the model whenever the underlying data changes."
)

role_prompt = f"Summarize the following in 3 bullet points for a non-technical executive:\n\n{article}"

show(compare_models(role_prompt, system=system_msg))

## 5. Structured Output (JSON)
*Task: Data extraction.* Requesting a strict schema for reliable, parseable output.

In [ ]:
structured_prompt = """Extract the following fields from this email as JSON with keys: sender_name, company, requested_product, quantity, deadline.
Respond with ONLY valid JSON, no explanation, no markdown code fences.

Email:
\"Hi, this is Maria Chen from Nordic Retail Group. We'd like to order 500 units of the SmartLock Pro. We need them delivered by September 15th. Thanks!\"
"""

results = compare_models(structured_prompt)
for name, text in results.items():
    print(f"--- {name} ---")
    try:
        print(json.dumps(json.loads(text), indent=2))
    except json.JSONDecodeError:
        print("(not valid JSON, raw output below)")
        print(text)
    print()

## 6. ReAct Pattern (Reason + Act)
*Task: Reasoning with tool use.* The model alternates Thought → Action → Observation, calling a `calculator` tool that we execute on its behalf and feed back as an observation. Expressions are evaluated with a restricted AST walker rather than `eval`, since the input is model-generated text.

In [ ]:
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}


def safe_eval(expr):
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expr}")
    return _eval(ast.parse(expr, mode="eval"))


REACT_INSTRUCTIONS = """Answer the question using this exact format:

Thought: reason about what to do next
Action: calculator[expression]
Observation: (this will be filled in by the system — do not write it yourself)
... repeat Thought/Action/Observation as needed ...
Final Answer: the answer

Only use the calculator Action when you need to compute something. Stop right after an Action line and wait for the Observation.

Question: {question}
"""


def run_react(question, ask_fn, max_steps=4):
    transcript = REACT_INSTRUCTIONS.format(question=question)
    for _ in range(max_steps):
        response = ask_fn(transcript)
        transcript += response
        if "Final Answer:" in response:
            break
        match = re.search(r"Action:\s*calculator\[(.*?)\]", response)
        if not match:
            break
        try:
            result = safe_eval(match.group(1))
        except Exception as e:
            result = f"Error: {e}"
        transcript += f"\nObservation: {result}\n"
    return transcript


question = (
    "A store buys widgets at $3.20 each and sells them at $5.75 each. "
    "If they sell 480 widgets, what is the total profit?"
)
print(run_react(question, ask_gpt))

## 7. Prompt Chaining
*Task: Summarization → transformation.* Break the task into sequential prompts, each building on the previous output.

In [ ]:
launch_announcement = (
    "Acme Robotics today announced the general availability of its new warehouse picking robot, Falcon-2. "
    "Falcon-2 can identify and pick over 1,200 SKUs per hour, a 40% improvement over the previous generation, "
    "while using 25% less power thanks to a redesigned battery management system. The company says "
    "early customers have cut picking errors by more than half. Falcon-2 ships starting next quarter, "
    "with pricing available on request."
)

summary = ask_gpt(f"Summarize this in 2 sentences:\n\n{launch_announcement}")
print("Summary:", summary)

tweet = ask_gpt(
    f"Turn this summary into a punchy tweet (under 280 characters) with 2 relevant hashtags:\n\n{summary}"
)
print("\nTweet:", tweet)

## 8. Code Generation Prompting
*Task: Code generation.* Ask for a function with an explicit contract: signature, docstring, and test cases.

In [ ]:
code_gen_prompt = (
    "Write a Python function `is_palindrome(s: str) -> bool` that checks if a string is a palindrome, "
    "ignoring case, spaces, and punctuation. Include a docstring and 3 doctest examples."
)

show(compare_models(code_gen_prompt))

## 9. Reflection / Iterative Refinement
*Task: Code generation.* The model critiques its own first draft, then produces a corrected version — catching edge cases a single pass tends to miss.

In [ ]:
first_pass = ask_gpt(code_gen_prompt)
print("--- First draft ---\n", first_pass)

critique_prompt = f"""Review this Python code for bugs, missed edge cases, and style issues. List concrete problems, then provide a corrected version.

Code:
{first_pass}"""

critique = ask_gpt(critique_prompt)
print("\n--- Critique + revision ---\n", critique)

## 10. Role-Playing / Persona Prompting
*Task: Text generation.* Assigning a persona in the prompt itself (rather than the system message) reshapes tone and style.

In [ ]:
persona_prompt = (
    "Rewrite this product description as if a very enthusiastic 1990s infomercial host is presenting it:\n\n"
    "Our wireless earbuds have 30-hour battery life, active noise cancellation, and a compact charging case."
)

show(compare_models(persona_prompt))

## 11. Self-Consistency
*Task: Reasoning.* Sample the same reasoning prompt multiple times and take the majority answer — more robust than trusting a single completion.

In [ ]:
question = "If a train leaves at 3:15pm and travels for 2 hours 40 minutes, what time does it arrive?"
prompt = f"{question}\nThink step by step, then give the final answer on its own line as 'Answer: HH:MM AM/PM'."

samples = [ask_gpt(prompt) for _ in range(5)]
for i, s in enumerate(samples, 1):
    print(f"Sample {i}:\n{s}\n")

answers = [m.group(1).strip() for m in (re.search(r"Answer:\s*(.+)", s) for s in samples) if m]
print("Answer distribution:", Counter(answers))
print("Majority answer:", Counter(answers).most_common(1)[0][0] if answers else "n/a")

## 12. Cross-Model Comparison
*Task: All task types.* The same prompt run through GPT, Claude, and a local Ollama model side by side, so differences in style, accuracy, and verbosity are easy to compare directly.

In [ ]:
import pandas as pd

comparison_prompt = (
    "Explain what a transformer's 'attention mechanism' does, in exactly 2 sentences, for a beginner."
)
results = compare_models(comparison_prompt)
df = pd.DataFrame(list(results.items()), columns=["Model", "Response"])
df

## Takeaways

_Fill this in after running the notebook with real API keys:_

- Which technique improved output quality the most for each task type?
- Where did GPT, Claude, and the local Ollama model disagree or differ in quality?
- Which technique was most worth the added prompt complexity, and which wasn't?